In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 16g: CNN Head — Real Only (eurosat)
# Uses pre-extracted MobileNetV2 features from notebook 13
# Hyperparameters fixed from notebook 16a HP search

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import classification_report
import gc

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

CLASSES    = 10
BATCH_SIZE = 128
EPOCHS     = 50
CLASS_NAMES = ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

# Fixed hyperparameters from notebook 16a HP search
ACTIVATION = 'relu'
OPTIMIZER  = 'adam'
DROPOUT    = 0.5

PROJECT_DIR       = '/content/drive/MyDrive/thesis_project'
DDIM_FEATURES_DIR = os.path.join(PROJECT_DIR, 'data', 'extracted_features', 'seed_42', 'ddim')
REAL_FEATURES_DIR = os.path.join(PROJECT_DIR, 'data', 'extracted_features', 'seed_42', 'real')
SYNTH_NPZ         = os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim', 'eurosat/eurosat_synthetic.npz')
REAL_NPZ          = os.path.join(PROJECT_DIR, 'data', 'processed', 'seed_42', 'eurosat.npz')
RESULTS_DIR       = os.path.join(PROJECT_DIR, 'results', 'cnn_head', 'eurosat', 'real_only')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Load Pre-extracted Features & Labels ──────────────────────────────

F_train_real = np.load(os.path.join(REAL_FEATURES_DIR, 'eurosat_x_train_features.npy'))
F_val_real   = np.load(os.path.join(REAL_FEATURES_DIR, 'eurosat_x_val_features.npy'))
F_train      = np.concatenate([F_train_real, F_val_real], axis=0)

real_data = np.load(REAL_NPZ)
y_train   = np.concatenate([real_data['y_train'], real_data['y_val']], axis=0)

F_test = np.load(os.path.join(REAL_FEATURES_DIR, 'eurosat_x_test_features.npy'))
y_test = real_data['y_test']

print(f'F_train: {F_train.shape}, y_train: {y_train.shape}')
print(f'F_test:  {F_test.shape},  y_test:  {y_test.shape}')

In [ ]:
# ── Cell 3: Model ─────────────────────────────────────────────────────────────

tf.keras.backend.clear_session()
gc.collect()

model = models.Sequential([
    layers.Input(shape=(1280,)),
    layers.Dense(64, activation=ACTIVATION),
    layers.Dropout(DROPOUT),
    layers.Dense(CLASSES, activation='softmax'),
], name='cnn_head')

model.compile(
    optimizer=optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Cell 4: Training ──────────────────────────────────────────────────────────

history = model.fit(
    F_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

pd.DataFrame(history.history).to_csv(
    os.path.join(RESULTS_DIR, 'history.csv'), index_label='epoch'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['accuracy'], label='train accuracy')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Training Curves — CNN Head Real Only (eurosat)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Training complete.')

In [ ]:
# ── Cell 5: Evaluation ────────────────────────────────────────────────────────

test_loss, test_acc = model.evaluate(F_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}')

y_pred = np.argmax(model.predict(F_test, verbose=0), axis=1)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

report_dict = classification_report(
    y_test, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True
)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(RESULTS_DIR, 'classification_report.csv')
)
pd.DataFrame([{
    'condition': 'real_only',
    'dataset':   'eurosat',
    'test_acc':  round(test_acc, 4),
    'test_loss': round(test_loss, 4),
}]).to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)
print(f'\nResults saved to: {RESULTS_DIR}')